In [1]:
# import libraries
import torch
import json
import random
import re
import sys
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import classification_report as sklearn_classification_report
from seqeval.metrics import classification_report as seqeval_classification_report

# import custom functions
sys.path.append("../utils")
from custom_evaluation import extract_spans, mention_level_evaluation

In [19]:
def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|\w+|'\w+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"\w+|'\w+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

# initialize empty dataset list
dataset = []

with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

# loop through all sentences in the data
for task in data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    dataset.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

In [5]:
# load the model
checkpoint = "HuggingFaceTB/SmolLM-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

In [44]:
# define chat template for normal inference
def compile_ner_prompt(few_shot_examples, test_sentence):
    chat = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that extracts *social group mentions* from text.\n\n"

                "### Definition of a Social Group\n"
                "A social group is a segment of society or a collection of people who share common socio-demographic traits "
                "or attributes that are either ascriptive (e.g., gender, ethnicity) or acquired (e.g., education, occupation). "
                "Relevant traits include sex, gender, age, ethnicity, language, religion, nationality, place of residence, "
                "income, occupation, and education.\n\n"

                "### Exclusions\n"
                "- Do **not** include implicit social group references such as *people*, *everyone*, *communities*, *the public*, or *the nation*.\n"
                "- Exclude **institutional or organizational entities** (e.g., trade unions, political parties, police departments, companies).\n"
                "- However, you **may include groupings within those institutions** if the defining feature is socio-demographic "
                "(e.g., *workers*, *union members*, *police officers*, *teachers*, *business owners*).\n"
                "- Exclude groupings defined primarily by shared beliefs, ideology, or political affiliation.\n\n"

                "### Task\n"
                "Identify and mark all social group mentions in the provided sentence.\n\n"

                "### Output Format\n"
                "- Return the full sentence.\n"
                "- Mark the **start** of each social group mention with `@@` and the **end** with `##`.\n"
                "- If there are no social group mentions, just respond with the full sentence without changing anything."
            )
        }
    ]

    # add few-shot examples
    for example in few_shot_examples:
        context = example["text"]
        answer = example["llm_text"]
        chat.append(
            {"role": "user", "content": f"Sentence: {context}"})
        chat.append({"role": "assistant", "content": answer})
    
    # add the test sentence
    chat.append(
        {"role": "user", "content": f"Sentence: {test_sentence}"})

    # compile the prompt
    prompt = tokenizer.apply_chat_template(
    chat, return_tensors="pt", tokenize=False, add_generation_prompt=True)
    return prompt

In [40]:
# create some few-shot examples
non_empty_examples = [ex for ex in dataset if ex["labels"]]
empty_examples = [ex for ex in dataset if not ex["labels"]]
few_shot_examples = random.sample(non_empty_examples, 4) + random.sample(empty_examples, 1)

# create test dataset
split_idx = int(len(non_empty_examples)*0.95)
test_dataset = non_empty_examples[split_idx:] + random.sample(empty_examples, int(len(empty_examples)*0.01))
random.shuffle(test_dataset)

In [41]:
# generate the answers for the normal format and store in a list
gen_answers = []

for i in range(len(test_dataset)):
    sentence = test_dataset[i]["text"]
    prompt = compile_ner_prompt(few_shot_examples, sentence)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**prompt_ids)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True)
    answer_text = generated_text.split("assistant\n")[-1]
    
    # convert the generated prediction to the bio scheme
    answer_bio = llm_output_to_bio(answer_text)
    #answer_list = to_list_or_empty(answer)
    gen_answers.append({"text": answer_text,
                        "bio": answer_bio})

In [43]:
# show a few predictions
for idx in range(10):
    input = test_dataset[idx]["text"]
    prediction = gen_answers[idx]["text"]
    print(f"Input: {input}")
    print(f"prediction: {prediction}")
    print("-"*100)

Input: Will this all-male Treasury team explain how that is helping families manage the cost-of-living crisis?
prediction: A4e recently pulled out of a £17 million contract to deliver education and training in London prisons.
----------------------------------------------------------------------------------------------------
Input: It is a serious crime and those who buy illegal cigarettes are supporting and funding evil criminals who are involved in significant violence.
prediction: It is a serious crime and those who buy illegal cigarettes are supporting and funding evil criminals who are involved in significant violence.
----------------------------------------------------------------------------------------------------
Input: The percentage of women on boards has increased to 17% from 12% since the election, and as I have said, a third of new appointments in the last year have been women.
prediction: The percentage of women on boards has increased to 17% from 12% since the election

In [16]:
# evaluate the generated answers

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in test_dataset]
pred_bio = [[tag for (_, tag) in sent["bio"]] for sent in gen_answers]

filtered_gt = []
filtered_pred = []
for gt, pred in zip(ground_truth_bio, pred_bio):
    if len(gt) == len(pred):
        filtered_gt.append(gt)
        filtered_pred.append(pred)

y_true = [tag for sent in filtered_gt for tag in sent]
y_pred = [tag for sent in filtered_pred for tag in sent]

# evaluate at the word level
print(sklearn_classification_report(y_true, y_pred))

# evaluate at the entity level with seqeval
print(seqeval_classification_report(filtered_gt, filtered_pred))

              precision    recall  f1-score   support

        B-sg       0.00      0.00      0.00        14
        I-sg       0.00      0.00      0.00        22
           O       0.91      1.00      0.95       365

    accuracy                           0.91       401
   macro avg       0.30      0.33      0.32       401
weighted avg       0.83      0.91      0.87       401

              precision    recall  f1-score   support

          sg       0.00      0.00      0.00        14

   micro avg       0.00      0.00      0.00        14
   macro avg       0.00      0.00      0.00        14
weighted avg       0.00      0.00      0.00        14



/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

In [38]:
all_true_spans = []
all_predicted_spans = []

for idx in range(len(filtered_gt)):

    # get the spans
    all_true_spans.append(extract_spans(filtered_gt[idx]))
    all_predicted_spans.append(extract_spans(filtered_pred[idx]))

# apply cross-span evaluation
mention_level_evaluation(all_true_spans, all_predicted_spans)

{'precision': 0.0, 'recall': 0.0, 'f1': 0.0}